# 4 · Publication lists, publications of each person, most-cited shares and venues

Needs notebooks 01 and 02 to have run.

In [ ]:
# Run this cell first. On Google Colab it installs the package; locally it does nothing
# (install once with `pip install -e .` from the project folder).
import sys

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/celoy/bibliometry"
    !pip install -q "git+{REPO_URL}"

In [ ]:
from bibliometry.env import project_root

ROOT = project_root()

In [ ]:
from bibliometry import pipeline
from bibliometry.config import Config

config = Config.load(ROOT / "config.yaml")

## The two publication lists

* **List 2 — people of the list**: publications where a person of your list signed **as a member of the institution**. Use this one to count the output of your people.
* **List 1 — institution**: every publication of the period with at least one author signing with the institution. In a joint lab, staff of a partner university also sign with the institution (their affiliation names both), so list 1 contains their papers too.

Both are saved with **every OpenAlex field** in `data/lists/` (`list2_roster.xlsx`, `list1_institution.xlsx`, and `.parquet` for Python), together with the venues (`venues.xlsx`), the indicators below (`indicators.xlsx`) and the publications of each person (`publications_by_person.xlsx`).

The first run also downloads a **world reference** of all active journals (~64,000, a few minutes, once).

In [ ]:
lists_out = pipeline.lists_step(config, pipeline.make_client(config))

## Publications of each person

`publications_by_person.xlsx` has one sheet per person: a short reference for each publication they signed with the institution (authors, title, journal, volume, issue, pages, DOI, type), most recent first. All types are kept (articles, proceedings, preprints, books, chapters, theses…); a paper and its preprint or HAL copy are listed once. A paper co-signed by two people of the list appears under both (`co_listed`).

The `summary` sheet gives, for each person, the indicators of the one-person notebook and the count per year. `signed_elsewhere` counts their works not signed with the institution: many of them can point to a wrong profile match.

In [ ]:
summary = lists_out["person_summary"]
summary

Type a `person_id` (first column above) to see that person's list:

In [ ]:
PERSON = ""  # @param {type:"string"}

pubs = lists_out["publications_by_person"]
pubs[pubs.person_id == PERSON][["publication_year", "authors", "title", "journal", "volume",
                                "issue", "pages", "doi", "type"]]

In [ ]:
from bibliometry.env import offer_download

for name in ("publications_by_person.xlsx", "list2_roster.xlsx", "indicators.xlsx"):
    offer_download(config.data_path / "lists" / name)

### Top 10% and 1% most cited

OpenAlex compares each publication with publications of the same field, year and type (`citation_normalized_percentile`). For an average institution, 10% of its publications are in the world's top 10%, and 1% in the top 1%. Shares are computed over publications that have a percentile (`coverage`); recent years are less reliable, as citations are still accumulating.

Two scopes: **all works**, and **research, deduplicated** (articles, reviews, conference papers, chapters, books, preprints, theses, reports; a paper and its preprint counted once).

In [ ]:
cited = lists_out["top_cited"]
cited.style.format({"coverage": "{:.0%}", "share_top10": "{:.1%}", "share_top1": "{:.1%}"})

#### The top-1% papers

Research outputs, deduplicated. In `authors`, `*` marks authors signing with the institution and `[person-id]` the people of your list.

In [ ]:
lists_out["top1_papers_list2"]

#### Whole institution (list 1, all signers)

The top-1% papers of every author signing with the institution, whether on your list or not.

In [ ]:
lists_out["top1_papers_list1"]

### Venues

Each publication's venue is its journal (or conference, book series…); a HAL/arXiv deposit is replaced by the journal when OpenAlex knows both.

**Headline metric — 2-year mean citedness** (OpenAlex's analogue of the impact factor: mean citations received in a year by the journal's papers of the two previous years). Two flags:

* `world_top10_2yr_mean_citedness` — the journal is in the top 10% of **all active journals in the world of the same main subfield** (`world_pct_…` = share of those journals doing better; `world_n` = their number). This compares like with like;
* `top10_2yr_mean_citedness` — in the top 10% of the journals used in list 1 (all fields together).

`broad_scope` marks journals spanning several domains (Scientific Reports, PLOS ONE, Nature…), whose main subfield is somewhat arbitrary.

**Secondary metrics — h-index and i10-index** (further right in the tables): they grow with a journal's size and age, so they say as much about volume as about impact.

In [ ]:
venues_table = lists_out["venues"]
print(f"{venues_table.attrs.get('n_ranked', venues_table.ranked.sum())} journals ranked")
venues_table[venues_table.ranked][
    ["venue_name", "main_subfield", "broad_scope", "works_list1", "works_list2",
     "2yr_mean_citedness", "world_pct_2yr_mean_citedness", "world_top10_2yr_mean_citedness",
     "top10_2yr_mean_citedness", "world_n"]
].head(30)

Share of publications (in ranked journals) that appeared in a top-10% journal — 2-year citedness first, then the secondary metrics:

In [ ]:
lists_out["top_venues"].T